In [ ]:
# Imports
from pathlib import Path

import echoregions as er
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from echograms import OKLab

from escore import (
    EscoreClassifier,
    ExtractionConfig,
    ExtractionResults,
    ReviewDecisions,
    WardExploration,
    apply_decisions,
    classify_sv,
    extract_echotypes,
    plot_review,
    sensitivity,
    summarize_candidates,
)
from escore.model import (
    plot_classes,
    plot_classification,
    plot_responses,
    plot_sensitivity,
)

In [ ]:
# User params
EXTRACTION_DIR: Path = Path("data/echotype-extraction/2D")  # Output directory
RUN_ID: str | None =  None                                  # (Optional) ID of a given extraction to load. If None, a new extraction is computed and saved.
NEW_REVIEW: bool = False
SV_PATH: str | Path = "data/input/Echointegration.zarr"     # Path to MVBS dataset
ROIS_PATH: str | Path = "data/rois-selection/rois.csv"      # Path to ROIs file

##  *Extraction* - Automatic ROI clustering

This step generates echotypes candidates.

In [ ]:
if RUN_ID is not None:
    extraction_path = EXTRACTION_DIR / f"{RUN_ID}/extraction.joblib"
    results = ExtractionResults.load(extraction_path)
    if results.extraction_id != RUN_ID:
        raise ValueError("Extraction ID does not match the requested run")
else:
    EXTRACTION_DIR.mkdir(parents=True, exist_ok=True)

    # Define config
    config = ExtractionConfig(
        channels=(38e3, 70e3, 120e3),
        kernel=np.ones((3, 3)),
        clusterer="gmm",
        confidence_threshold=0.75,
    )

    # Load input data
    da_Sv = xr.open_dataset(SV_PATH, engine="zarr").Sv
    regions = er.read_regions_csv(ROIS_PATH)

    # Extract & persist
    results = extract_echotypes(da_Sv, regions, config=config)
    run_id = results.extraction_id

    save_dir = EXTRACTION_DIR / run_id
    save_dir.mkdir()
    extraction_path = save_dir / "extraction.joblib"
    results.save(extraction_path)

    print(f"ExtractionResults saved at {extraction_path}")


In [ ]:
print(
    f"Run id: {results.extraction_id}"
    "\nResults:"
    f"\n * {len(results.candidates)} echotypes candidates"
    f"\n * {len(results.failures)} failures"
    f"\n * Total: {len(results.candidates) + len(results.failures)}"
)

## *Review* - Accept / edit echotypes 

This step is interactive: the user loop through echotype candidates, and make decisions. For 

In [ ]:
decisions_path = EXTRACTION_DIR / f"{results.extraction_id}/decisions.json"
if "decisions" in locals():
    if decisions.extraction_id != results.extraction_id:
        raise ValueError("Existing in-memory review belongs to a different extraction")
elif NEW_REVIEW:
    if decisions_path.exists():
        raise FileExistsError(f"Review already exists: {decisions_path}")
    decisions = ReviewDecisions.new(results)
elif decisions_path.exists():
    decisions = ReviewDecisions.load(decisions_path)
    if decisions.extraction_id != results.extraction_id:
        raise ValueError("Review belongs to a different extraction")
else:
    decisions = ReviewDecisions.new(results)

In [ ]:
ROI_START = 1 # (Optional) Set higher ROI id (e.g. to start from the last reviewed ROI)

# Iterator to easily loop through candidate echotypes with next()
candidates_loop = iter({k: v for k, v in results.candidates.items() if k >= ROI_START}.values())

In [ ]:
# Loop (run again to go the next candidate)
c = next(candidates_loop)

# Or select candidates for a given ROI
# c = get_candidate(results, roi_id=10)

### Inspect extraction result

The `plot_review` function displays an inspection figure for a clustered ROI. 

By default (`CID = None`), the next cell will display the estimated best cluster. You can explore the distribution and mask of the other clusters by manually setting `CID`.

In [ ]:
# Edit to view a cluster (if None, the proposed cluster is shown)
CID: int | None = None

# Leave this part untouched
cid = CID if CID is not None else c.proposed_cluster
print(f"User defined typology class: {c.metadata['region_name']}")
_ = plot_review(results, c.roi_id, cluster_id=cid, rgb_encoder=OKLab())

### Make a review decision

Based on the inspection plots, you must now make a decision about the current ROI. Available decisions are (corresponding `ReviewDecisions` methods are detailed):

1. Select one (or more) clusters as echotypes → use `.select(roi_id, cluster_id)` (with valid `cluster_id`)
2. Reject the entire ROI → use `.reject_roi(roi_id)`

By default all new decisons remain with status "pending". You need to accept a decision to make it effective. This is done using `.accept_roi(roi_id)` method.

**Note:** Once an ROI is accepted (or rejected), the corresponding decisions are locked. To modify them, you must first reset the status to "pending" by using `.edit_roi(roi_id)`.

In [ ]:
# Print current decisions for this ROI (and their status)
decisions.summary(c.roi_id)

In [ ]:
raise SystemExit("Blocking downstream execution - User must run the following cells manually")

To make the proposed decision effective, you must raise its status to "accepted" using `.accept_roi(roi_id)`. Only after can you select another cluster (the accept it, *etc*).

In [ ]:
decisions.accept_roi(c.roi_id)  # Accept the ROI's decisions

Run the following cell to select one or more other cluster as echotypes (you need to validate with `.accept_roi(roi_id)` after each new selection)

In [ ]:
decisions.select(c.roi_id, cluster_id=2)  # Select a different cluster

If the ROI's clusters are overall poor quality. You may reject the ROI altogether using `.reject_roi(roi_id)`. However, this will raise if at least one cluster has been accepted: Use `.edit_roi(roi_id)` to go back to "pending" status before rejecting.

In [ ]:
decisions.reject_roi(c.roi_id, reason="Non-normal R(f)")  # Drop this ROI entirely

## Inspect the review

In [ ]:
decisions_path = EXTRACTION_DIR / f"{decisions.extraction_id}/decisions.json"
decisions.save(decisions_path)
decisions = ReviewDecisions.load(decisions_path)

In [ ]:
summarize_candidates(results, decisions)

## Apply decisions and fetch echotypes

In [ ]:
outcome = apply_decisions(results, decisions, allow_pending=True)
echotypes = outcome.echotypes

print(
    "Outcome of the echotype review process:"
    f"\n * {len(outcome.echotypes.items)} echotypes created"
    f"\n * {len(outcome.failures)} failed extractions"
    f"\n * {len(outcome.rejected)} rejected ROIs"
    f"\n * {len(outcome.pending)} pending decision (ignored)"
)

In [ ]:
mean_delta_df = echotypes.means
df1 = mean_delta_df.iloc[:, 0].values
df2 = mean_delta_df.iloc[:, 1].values


fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
ax.scatter(df1, df2, s=5)
ax.set_axisbelow(True)
ax.grid()
ax.set_aspect(1)
ax.set_xlim(-20, 20)
ax.set_ylim(-20, 20)
ax.set_xlabel(r"$\Delta \text{MVBS}_{70 - 38 \text{ kHz}} \text{ [dB]}$")
ax.set_ylabel(r"$\Delta \text{MVBS}_{120 - 38 \text{ kHz}} \text{ [dB]}$")
ax.set_title("Echotypes Means");

## Echoclass model
Review the unresolved ROIs before training. Ward scores are descriptive; choose the number of classes after inspecting them and the dendrogram.

In [ ]:
# Require resolved review; use allow_pending=True only for an intentional partial library.
outcome = apply_decisions(results, decisions, allow_pending=True)
echotypes = outcome.echotypes
exploration = WardExploration(echotypes.means)
display(exploration.scores(tuple(range(2, min(10, len(echotypes.items) - 1) + 1))))
exploration.plot();

In [ ]:
# Select these explicitly after inspection; no universal distance threshold is assumed.
N_CLASSES = 6  # Set after inspecting the scores and dendrogram.
MAX_SQUARED_DISTANCE = None
if N_CLASSES is None:
    raise ValueError("Choose N_CLASSES before fitting")
model = EscoreClassifier(n_classes=N_CLASSES, max_squared_distance=MAX_SQUARED_DISTANCE)
model.fit(echotypes.means, channels=results.config.channels)
model.save(EXTRACTION_DIR / results.extraction_id / "escore-model.joblib")
training_classes = pd.Series(model.labels_, index=echotypes.means.index)

In [ ]:
_ = plot_classes(model, echotypes.means)
ax = plt.gca()
ax.set_aspect(1)
ax.set_xlim(-20, 20)
ax.set_ylim(-20, 20)

In [ ]:
_ = plot_responses(model)

In [ ]:
# Training coherence, not independent validation. Evaluate held-out ROIs separately.
detail, by_class = sensitivity(model, echotypes, training_classes, list(np.linspace(1, 31)))
_ = plot_sensitivity(by_class)

In [ ]:
# Set a threshold informed by sensitivity analysis before classifying a survey.
# model.max_squared_distance = ...
sv = xr.open_dataset(SV_PATH, engine="zarr").Sv
classification = classify_sv(model, sv)
# classification.to_netcdf(EXTRACTION_DIR / results.extraction_id / 'classification.nc')